# Investigation 1: Adjacency Signal (Q1)

**Hypothesis:** Companies that hold an active lease on a block adjacent to an available block bid on that available block at a meaningfully higher rate than companies without an adjacent position.

**Decision Rule:**
- Lift >= 3x, p < 0.05 → Strong signal. Proceed to MVP 1 as planned.
- Lift 1.5x–3x, p < 0.05 → Moderate signal. Proceed but reweight adjacency.
- Lift < 1.5x or p > 0.05 → Weak signal. Do not proceed as written; test planning-area aggregation.

**Validation discipline:** Features are built from Sales 257 and 261 only. December 2025 is held-out — used only in Section 5.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2_contingency
from libpysal.weights import Queen
import os
import re

# Paths
SHAPEFILE_PATH = '../../data/shapefiles/blocks.shp'
SALE_257_DIR = '../data/sale_257/'
SALE_261_DIR = '../data/sale_261/'
SALE_DEC2025_DIR = '../data/sale_obbba_dec2025/'
LEASE_PATH = '../data/leases/active_leases.csv'

print('Paths configured.')

In [ ]:
# Load shapefile
gdf_blocks = gpd.read_file(SHAPEFILE_PATH)
print(f'Loaded {len(gdf_blocks)} blocks.')
print(f'CRS: {gdf_blocks.crs}')
print(f'Columns: {list(gdf_blocks.columns)}')
gdf_blocks.head()

In [ ]:
# --- Parser functions (adapted from utils/process-lease-sale-BOEM-downloads.py) ---

def numeric_only(val):
    """Removes 'G' prefixes and non-digit characters."""
    if pd.isna(val): return val
    return re.sub(r'\D', '', str(val)).strip()

def load_bid_file(data_dir):
    """Load the BID file from a sale directory."""
    bid_file = [f for f in os.listdir(data_dir) if f.upper().endswith('.BID')][0]
    bid_specs = [(0, 7), (8, 15), (16, 26), (27, 32), (35, 43)]
    bid_names = ['Sale_Number', 'Lease_Number', 'Bid_Amount', 'Company_Number', 'Bid_Percentage']
    df = pd.read_fwf(os.path.join(data_dir, bid_file), colspecs=bid_specs, names=bid_names)
    df['Lease_Number'] = df['Lease_Number'].apply(numeric_only).str.zfill(7)
    df['Company_Number'] = df['Company_Number'].astype(str).str.strip().str.zfill(5)
    df['Bid_Amount'] = pd.to_numeric(df['Bid_Amount'], errors='coerce')
    return df

def load_trt_file(data_dir):
    """Load the TRT (tract) file from a sale directory."""
    trt_file = [f for f in os.listdir(data_dir) if f.upper().endswith('.TRT')][0]
    # TODO: Adjust colspecs after inspecting actual TRT file format
    df = pd.read_fwf(os.path.join(data_dir, trt_file))
    return df

def load_com_file(data_dir):
    """Load the COM (company) file from a sale directory."""
    com_file = [f for f in os.listdir(data_dir) if f.upper().endswith('.COM')][0]
    # TODO: Adjust colspecs after inspecting actual COM file format
    df = pd.read_fwf(os.path.join(data_dir, com_file))
    return df

def load_res_file(data_dir):
    """Load the RES (results) file from a sale directory."""
    res_file = [f for f in os.listdir(data_dir) if f.upper().endswith('.RES')][0]
    # TODO: Adjust colspecs after inspecting actual RES file format
    df = pd.read_fwf(os.path.join(data_dir, res_file))
    return df

print('Parser functions defined.')

In [ ]:
# Load bid data for all three sales
# TODO: Uncomment when data is downloaded
# df_bid_257 = load_bid_file(SALE_257_DIR)
# df_bid_261 = load_bid_file(SALE_261_DIR)
# df_bid_dec2025 = load_bid_file(SALE_DEC2025_DIR)

# Load active lease data
# df_leases = pd.read_csv(LEASE_PATH)

print('Data loading cells ready. Uncomment when BOEM files are downloaded.')

## 2. Build Block Adjacency Matrix

Queen contiguity: two blocks are adjacent if they share a full edge (not just a corner point).
We use `libpysal.weights.Queen` for this.

In [ ]:
# Build queen contiguity weights
# w = Queen.from_dataframe(gdf_blocks)
# print(f'Adjacency matrix built: {w.n} blocks, mean {w.mean_neighbors:.1f} neighbors')

# Spot-check: pick a known block and verify neighbors
# test_idx = gdf_blocks[gdf_blocks['AC_LAB'] == 'MC127'].index[0]
# neighbor_indices = w.neighbors[test_idx]
# print(f'MC127 has {len(neighbor_indices)} neighbors:')
# gdf_blocks.loc[neighbor_indices, ['PROT_NUMBE', 'BLOCK_NUMB', 'AC_LAB']]

## 3. Build Active Lease Lookup

For each company, identify which blocks they hold active leases on as of each sale's bid deadline.
Cross-reference with adjacency matrix to determine which companies have adjacent active leases for each available block.

In [ ]:
# TODO: Implement when lease data is available
# Steps:
# 1. Filter leases active as of Sale 257 bid deadline (approx. Aug 2023)
# 2. Create a dict: {company_id: set(block_ids)} for active leases
# 3. For each available block in Sale 257 TRT file:
#    - Get its queen-adjacent blocks from the adjacency matrix
#    - For each company, check if they hold a lease on any adjacent block
#    - Flag: adjacent = 1 if yes, 0 if no
# 4. Repeat for Sale 261
pass

## 4. Training Sanity Check (Sales 257 / 261)

**This is NOT validation.** This is a sanity check that the adjacency variable is constructed correctly.
If lift is exactly 1.0, something is wrong with the join.

In [ ]:
# TODO: Compute bid_rate_adjacent and bid_rate_non_adjacent on Sales 257/261
# Sanity check: lift should be meaningfully > 1.0
pass

---
## 5. VALIDATION — HELD-OUT (December 2025 Sale)

**⚠️ This section uses the held-out December 2025 sale for evaluation only.**
**Features (adjacency flags) are computed from Sales 257/261 lease data. Only bid outcomes are from Dec 2025.**

In [ ]:
# TODO: Compute adjacency-based metrics on Dec 2025 held-out data
#
# 1. For each available block in Dec 2025 TRT:
#    - Check which companies have adjacent active leases (from lease data as of Dec 2025 bid deadline)
#    - NOTE: The lease STATUS is observable pre-sale. Only the BID OUTCOME is held-out.
#
# 2. Compute:
#    bid_rate_adjacent = bids_placed_by_adjacent_companies / total_adjacent_opportunities
#    bid_rate_non_adjacent = bids_placed_by_non_adjacent_companies / total_non_adjacent_opportunities
#    lift = bid_rate_adjacent / bid_rate_non_adjacent
#
# 3. Build 2x2 contingency table:
#              | Bid | No Bid |
#    Adjacent  |  a  |   b    |
#    Not Adj   |  c  |   d    |
#
# 4. Chi-square test
#    chi2, p, dof, expected = chi2_contingency([[a, b], [c, d]])
pass

In [ ]:
# Print results
# print(f'Bid rate (adjacent):     {bid_rate_adjacent:.4f}')
# print(f'Bid rate (non-adjacent): {bid_rate_non_adjacent:.4f}')
# print(f'Lift:                    {lift:.2f}x')
# print(f'Chi-square:              {chi2:.2f}')
# print(f'p-value:                 {p:.6f}')
# print(f'Significant (p < 0.05):  {p < 0.05}')

## 6. Choropleth Map

GOM grid colored by "number of companies with adjacent active lease" for Dec 2025 available blocks.
Actual Dec 2025 bids overlaid as colored dots (by company).

In [ ]:
# TODO: Build choropleth
# fig, ax = plt.subplots(1, 1, figsize=(20, 12))
# ... plot adjacency density as a heatmap ...
# ... overlay actual bids as scatter points ...
# ax.set_title('Q1: Adjacency Signal — Dec 2025 Sale (Held-Out Validation)')
# plt.tight_layout()
# plt.savefig('../outputs/q1_adjacency_choropleth.png', dpi=150, bbox_inches='tight')
pass

## 7. Findings

### Results
- **Lift:** [TBD]x
- **p-value:** [TBD]
- **Contingency table:** [TBD]

### Interpretation
[TBD — fill in based on results and PRD decision rule]

### Recommendation
[TBD — Proceed / Proceed with modifications / Pause and reframe]